# METR Estimates: Create Excel file with all relevant numbers

### Load packages & set year and paths

In [237]:
# import packages
import pickle
import pandas as pd
import pickle
import tjn_tools
import numpy as np
import openpyxl

# Set path and year for analysis
path = "C:/Users/AlisonSchultz/Tax Justice Network Ltd/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/METR/2023"
year = 2018

### 1. Import OECD's CbCR data
- from: https://stats.oecd.org/Index.aspx?DataSetCode=CBCR_TABLEI
- use .csv file which includes data on all available years

In [238]:
# Import CbCR data
df = pd.read_csv("{}/2_data/1_input/Table1_CbCR.csv".format(path),
                 usecols=["COU","JUR","Partner Jurisdiction","Grouping","Variable","Value","Year"])

## 2. Bring data in correct format
- Create ETR for each jurisdiction
- Obtain CIT for each jurisdiction
- Create total employees and total revenue for each jurisdiction

In [239]:
# Define which columns we need for analysis
fin_needed = ['Unrelated Party Revenues','Profit (Loss) before Income Tax',
       'Income Tax Paid (on Cash Basis)','Income Tax Accrued - Current Year','Number of Employees',
       'Tangible Assets other than Cash and Cash Equivalents', 'Year']

# Calculate ETRs
df = df.loc[(df["Grouping"]=="Sub-Groups with positive profits") & (df["Year"]==year) ]
df = df.loc[df["Variable"].isin(fin_needed)]
df = pd.pivot_table(df,index=["COU","JUR","Partner Jurisdiction", "Year"],values="Value",columns="Variable").reset_index() 

#Financial variables
df_fin = df.loc[:,["COU","JUR","Number of Employees","Profit (Loss) before Income Tax","Income Tax Paid (on Cash Basis)","Unrelated Party Revenues","Tangible Assets other than Cash and Cash Equivalents"]]
df_fin.columns = ["iso3_o","iso3_d","emp","pi","txc","revt","assets"]
df_fin["Domestic"] = (df_fin["iso3_o"]==df_fin["iso3_d"]).astype(int)

# Add statutory income tax rate from Javier's file
iso3_to_cit = pickle.load(open(f"{path}/2_data/1_input/{year}_iso3_to_cit.dump".format(path),"rb+"))

# Create dictionaries with employees, sales, and ETR per jurisdiction
iso3_to_emp = df_fin.groupby(["iso3_d"]).sum(numeric_only=True).to_dict()["emp"]
iso3_to_revt = df_fin.groupby(["iso3_d"]).sum(numeric_only=True).to_dict()["revt"]
iso3_to_pi = df_fin.groupby(["iso3_d"]).sum(numeric_only=True).to_dict()["pi"]
iso3_to_txc = df_fin.groupby(["iso3_d"]).sum(numeric_only=True).to_dict()["txc"]
iso3_to_assets = df_fin.groupby(["iso3_d"]).sum(numeric_only=True).to_dict()["assets"]
iso3_to_etr = df_fin.groupby(["iso3_d"])[["pi", "txc"]].sum(numeric_only=True)
iso3_to_etr["etr"] = iso3_to_etr["txc"] / iso3_to_etr["pi"]
iso3_to_etr.loc[(iso3_to_etr["pi"] == 0), "etr"] = np.nan
# replace ETR by CIT if ETR cannot be calculated (following Javier)
iso3_to_etr["cit"] = iso3_to_etr.index.map(iso3_to_cit).fillna(None)
iso3_to_etr.loc[iso3_to_etr["etr"]<=0.0001,"etr"] = iso3_to_etr.loc[iso3_to_etr["etr"]<=0.0001,"cit"]
iso3_to_etr['etr'].fillna(iso3_to_etr["cit"], inplace=True)
iso3_to_etr.loc[iso3_to_etr["etr"]>0.6,"etr"] = 0.6 # Following Javier: Add a max (to avoid weird phenomena in the regressions)
iso3_to_etr = iso3_to_etr.to_dict()["etr"]
# check my ETRs against the ones created by Javier
iso3_to_etr_javier = pickle.load(open("{}/2_data/z_archive/iso3_to_etr.dump".format(path),"rb+"))

## 3. Create tables with country specific info on ETR, employees, and revenues
- Country groups are defined in Model_OECD_data_updated.xlsx, sheet "Countries"


In [240]:
# Import country groups (finer classification in 13 groups and coarser classification in 4 groups)
country_groups13 = pd.read_excel(f"../2_data/1_input/Country_grouping_{year}.xlsx",sheet_name="13_groups")
country_groups4 = pd.read_excel(f"../2_data/1_input/Country_grouping_{year}.xlsx",sheet_name="4_groups")
# Substitute country names with ISO 3
country_groups13 = country_groups13.applymap(lambda x: tjn_tools.name_to_iso3(x.replace("\n"," ")) if isinstance(x,str) else np.nan)
country_groups4 = country_groups4.applymap(lambda x: tjn_tools.name_to_iso3(x.replace("\n"," ")) if isinstance(x,str) else np.nan)

In [241]:
# Create tables 
## for finer classification of countries in 13 groups
country_groups_emp13 = country_groups13.applymap(lambda x: iso3_to_emp[x] if iso3_to_emp.get(x) is not None else 0)
country_groups_revt13 = country_groups13.applymap(lambda x: iso3_to_revt[x] if iso3_to_revt.get(x) is not None else 0)
country_groups_pi13 = country_groups13.applymap(lambda x: iso3_to_pi[x] if iso3_to_pi.get(x) is not None else 0)
country_groups_txc13 = country_groups13.applymap(lambda x: iso3_to_txc[x] if iso3_to_txc.get(x) is not None else 0)
country_groups_assets13 = country_groups13.applymap(lambda x: iso3_to_assets[x] if iso3_to_assets.get(x) is not None else 0)
country_groups_etr13 = country_groups13.applymap(lambda x: iso3_to_etr[x] if iso3_to_etr.get(x) is not None else None)
country_groups_etr_javier13 = country_groups13.applymap(lambda x: iso3_to_etr_javier[x] if iso3_to_etr.get(x) is not None else None)
country_groups_cit13 = country_groups13.applymap(lambda x: iso3_to_cit[x] if iso3_to_cit.get(x) is not None else None)
## for coarser classification in countries in 4 groups
country_groups_emp4 = country_groups4.applymap(lambda x: iso3_to_emp[x] if iso3_to_emp.get(x) is not None else 0)
country_groups_revt4 = country_groups4.applymap(lambda x: iso3_to_revt[x] if iso3_to_revt.get(x) is not None else 0)
country_groups_pi4 = country_groups4.applymap(lambda x: iso3_to_pi[x] if iso3_to_pi.get(x) is not None else 0)
country_groups_txc4 = country_groups4.applymap(lambda x: iso3_to_txc[x] if iso3_to_txc.get(x) is not None else 0)
country_groups_assets4 = country_groups4.applymap(lambda x: iso3_to_assets[x] if iso3_to_assets.get(x) is not None else 0)
country_groups_etr4 = country_groups4.applymap(lambda x: iso3_to_etr[x] if iso3_to_etr.get(x) is not None else None)
country_groups_etr_javier4 = country_groups4.applymap(lambda x: iso3_to_etr_javier[x] if iso3_to_etr.get(x) is not None else None)
country_groups_cit4 = country_groups4.applymap(lambda x: iso3_to_cit[x] if iso3_to_cit.get(x) is not None else None)

In [242]:
# Export tables to Excel
## for finer classification of countries in 13 groups
workbook = openpyxl.load_workbook(f"{path}/2_data/2_output/{year}/METR_estimates_13groups{year}.xlsx")
worksheet_name = 'Countries_emp'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=5)
country_groups_emp13_list = country_groups_emp13.values.tolist()
new_sheet.append(columns)
for row in country_groups_emp13_list:
    new_sheet.append(row)

worksheet_name = 'Countries_etr'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=6)
country_groups_etr13_list = country_groups_etr13.values.tolist()
new_sheet.append(columns)
for row in country_groups_etr13_list:
    new_sheet.append(row)

worksheet_name = 'Countries_cit'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=7)
country_groups_cit13_list = country_groups_cit13.values.tolist()
new_sheet.append(columns)
for row in country_groups_cit13_list:
    new_sheet.append(row)

worksheet_name = 'Countries_revt'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=8)
country_groups_revt13_list = country_groups_revt13.values.tolist()
new_sheet.append(columns)
for row in country_groups_revt13_list:
    new_sheet.append(row)

workbook.save(f"{path}/2_data/2_output/{year}/METR_estimates_13groups{year}.xlsx".format(path))


## for coarser classification of countries in 4 groups
workbook = openpyxl.load_workbook(f"{path}/2_data/2_output/{year}/METR_estimates_4groups{year}.xlsx")
worksheet_name = 'Countries_emp'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=5)
country_groups_emp4_list = country_groups_emp4.values.tolist()
new_sheet.append(columns)
for row in country_groups_emp4_list:
    new_sheet.append(row)

worksheet_name = 'Countries_etr'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=6)
country_groups_etr4_list = country_groups_etr4.values.tolist()
new_sheet.append(columns)
for row in country_groups_etr4_list:
    new_sheet.append(row)

worksheet_name = 'Countries_cit'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=7)
country_groups_cit4_list = country_groups_cit4.values.tolist()
new_sheet.append(columns)
for row in country_groups_cit4_list:
    new_sheet.append(row)

worksheet_name = 'Countries_revt'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=7)
country_groups_revt4_list = country_groups_revt4.values.tolist()
new_sheet.append(columns)
for row in country_groups_revt4_list:
    new_sheet.append(row)

worksheet_name = 'Countries_pi'
worksheet = workbook[worksheet_name]
columns = [cell.value for cell in worksheet[1]]
workbook.remove(worksheet)
new_sheet = workbook.create_sheet(worksheet_name, index=7)
country_groups_pi4_list = country_groups_pi4.values.tolist()
new_sheet.append(columns)
for row in country_groups_pi4_list:
    new_sheet.append(row)

workbook.save(f"{path}/2_data/2_output/{year}/METR_estimates_4groups{year}.xlsx".format(path))

## 4. Create group level estimates of average ETR

In [243]:
# FINE Classification
# substitute nans in ETR with zero to be able to sum over ETRs
etr_values13 = country_groups_etr13.values
etr_values13[np.isnan(etr_values13)] = 0
# take employment-weighted mean of ETR per country group
etrs13 = (country_groups_emp13.values*etr_values13).sum(0)/country_groups_emp13.sum().values
groups13 = country_groups13.columns
groups13 = [_.replace("\n"," ") for _ in groups13]
len(etrs13),len(groups13) # check if both have the correct length

(14, 14)

In [244]:
# COARSE Classification
# substitute nans in ETR with zero to be able to sum over ETRs
etr_values4 = country_groups_etr4.values
etr_values4[np.isnan(etr_values4)] = 0
# take employment-weighted mean of ETR per country group
etrs4 = (country_groups_emp4.values*etr_values4).sum(0)/country_groups_emp4.sum().values
groups4 = country_groups4.columns
groups4 = [_.replace("\n"," ") for _ in groups4]
len(etrs4),len(groups4) # check if both have the correct length

(4, 4)

In [245]:
# create dictionary with ETR on group level
group_to_etr13 =  dict(zip(groups13,etrs13)) 
group_to_etr13 # check if ETRs look okay

{'A. Americas - High income': 0.08752187175218164,
 'B. Europe & Central Asia - High income': 0.12921322161001142,
 'C. East Asia & Pacific - High income': 0.19670304886257395,
 'D. Middle East & North Africa - High income': 0.3352504128675128,
 'E. Latin Am. & Caribbean - Middle and low income': 0.19067837543100594,
 'F. Europe & Central Asia - Middle and low income': 0.20159333769233434,
 'G. East Asia & Pacific - Middle and low income': 0.19871378913358773,
 'H. Middle East & North Africa - Middle and low income': 0.25840182671357675,
 'I. South Asia - Middle and low income': 0.32560261538176644,
 'J. Sub-Saharan - High and middle income': 0.14781350068257196,
 'K. Sub-Saharan - Low income': 0.2860002340878303,
 'L. Americas investment hubs': 0.01005631154035635,
 'M. European investment hubs': 0.0651119674841519,
 'N. Other investment hubs': 0.06320046257247293}

In [246]:
group_to_etr4 =  dict(zip(groups4,etrs4)) 
group_to_etr4

{'High income': 0.12767461708098538,
 'Middle Income': 0.2199252658712607,
 'Low Income': 0.3018692739694614,
 'Investment Hubs': 0.06425583691604032}